# Track training (local)

Reads the JSONL metrics sidecar that `train_pipeline/train_head.py` writes next to its `--output` (e.g. `trained_heads/mlp/model.metrics.jsonl`) and plots train/val curves — no wandb cloud connection needed.

Each line is one logged row: `train/*` rows on a step cadence, `val/*` rows once per epoch. Run the **live follow** cell to watch a run as it trains.

In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output

# Point this at the run you want to track (relative to repo root).
METRICS_PATH = Path("../trained_heads/mlp/model.metrics.jsonl")


def load_metrics(path):
    """Read the JSONL sidecar into a DataFrame, tolerating a half-written
    final line (the trainer may be mid-flush when we read)."""
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue  # partial last line during a live write
    return pd.DataFrame(rows)


def plot_metrics(df):
    """Plot mean MSE and mean PCC for train (step cadence) and val (per epoch)."""
    fig, (ax_mse, ax_pcc) = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric in [(ax_mse, "mean_mse"), (ax_pcc, "mean_pcc")]:
        for split, marker in [("train", ""), ("val", "o")]:
            col = f"{split}/{metric}"
            if col not in df.columns:
                continue
            sub = df.dropna(subset=[col])
            if len(sub):
                ax.plot(sub["step"], sub[col], marker=marker, label=split)
        ax.set(xlabel="step", ylabel=metric, title=metric)
        ax.legend()
    plt.tight_layout()
    plt.show()

## Static snapshot
Load and plot whatever has been logged so far.

In [ ]:
df = load_metrics(METRICS_PATH)
print(f"{len(df)} rows · steps {df['step'].min()}-{df['step'].max()}")
plot_metrics(df)
df.tail()

## Live follow
Re-reads the file and redraws every few seconds. Interrupt the kernel (■ / `Ctrl-C`) to stop.

In [ ]:
REFRESH_SEC = 5

try:
    while True:
        df = load_metrics(METRICS_PATH)
        clear_output(wait=True)
        plot_metrics(df)
        last = int(df["step"].max()) if len(df) else None
        print(f"{len(df)} rows · last step {last} · refreshing every {REFRESH_SEC}s (interrupt to stop)")
        time.sleep(REFRESH_SEC)
except KeyboardInterrupt:
    print("stopped following")